In [1]:
import gradio as gr
import requests
import os
import json
import sqlite3
import requests
import datetime
from ddgs import DDGS

api_key = "your_api_key_here"
url = "https://api.deepseek.com/chat/completions"

# 1. 告诉大模型，它拥有什么工具，以及工具的格式
tools_description = """
你可以使用以下工具来辅助回答。如果你需要使用工具，请仅输出一个特定的 JSON 格式，不要输出其他任何内容。
可用工具：
1. 查时间：格式 {"tool": "get_time", "args": {}}
2. 查南宁天气：格式 {"tool": "get_weather", "args": {}}
3. 记账：格式 {"tool": "add_expense", "args": {"item": "物品名称", "amount": 金额数字}}
4. 查总花费：格式 {"tool": "get_total_expense", "args": {}}
5. 查账本明细：格式 {"tool": "get_expense_list", "args": {}}
6. 查询本地知识库：当用户询问公司规定、南宁美食、AI历史等特定信息时，请使用此工具。格式 {"tool": "search_knowledge", "args": {"query": "搜索关键词"}}
如果不需要工具就能回答，请直接正常回答。
"""
def get_expense_list():
    conn = sqlite3.connect("my_ledger.db")
    cursor = conn.cursor()
    cursor.execute("SELECT item, amount FROM expenses")
    records = cursor.fetchall()
    conn.close()
    
    if not records:
        return "还没有任何记账记录。"
    
    result = "你的账本明细如下：\n"
    for item, amount in records:
        result += f"- {item}：{amount} 元\n"
    return result

def execute_tool(tool_name, args):
    if tool_name == "get_time":
        import datetime
        now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        return f"当前时间是：{now}"
        
    elif tool_name == "get_weather":
        return get_real_weather()
        
    elif tool_name == "add_expense":
        item = args.get("item", "")
        amount = args.get("amount", 0)
        return add_expense(item, amount)
        
    elif tool_name == "get_total_expense":
        return get_total_expense()
        
    elif tool_name == "get_expense_list":
        return get_expense_list()
        
    elif tool_name == "search_knowledge":
        return search_knowledge(args.get("query", ""))
        
    return "未知工具"
# 🌟 知识库检索工具（简单版 RAG）
def search_knowledge(query):
    if not os.path.exists("knowledge.txt"):
        return "未找到知识库文件。"
    
    with open("knowledge.txt", "r", encoding="utf-8") as f:
        content = f.read()
        
    paragraphs = content.split("\n") # 按换行切分
        
    results = []
    for p in paragraphs:
        # 🌟 升级版匹配：只要你问的问题里，有任何两个字出现在段落里，就算命中！
        # 比如你问“迟到怎么扣钱”，“迟到”和“扣钱”在段落里有，就会命中。
        match_count = 0
        for i in range(len(query) - 1):
            if query[i:i+2] in p: # 提取问题里的两个字作为关键词
                match_count += 1
                
        if match_count > 0: # 只要有两个字对上了，就抓取这段
            results.append(p)
            
    if results:
        return "在知识库中找到了以下相关内容：\n" + "\n".join(results[:3])
    else:
        return "知识库中没有找到相关内容。"
def init_db():
    conn = sqlite3.connect("my_ledger.db")
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS expenses
                      (id INTEGER PRIMARY KEY AUTOINCREMENT, 
                       item TEXT, amount REAL)''')
    conn.commit()
    conn.close()

init_db() # 启动时初始化

# 🌟 2. 真实的天气 API 逻辑（南宁的经纬度）
def get_real_weather():
    try:
        # 用免费的 Open-Meteo 接口查南宁天气
        url = "https://api.open-meteo.com/v1/forecast?latitude=22.82&longitude=108.32&current_weather=true"
        res = requests.get(url, timeout=5)
        data = res.json()
        temp = data["current_weather"]["temperature"]
        wind = data["current_weather"]["windspeed"]
        return f"南宁当前气温 {temp}°C，风速 {wind} km/h。"
    except Exception as e:
        return f"天气查询失败：{e}"

# 🌟 3. 真实的数据库记账逻辑
def add_expense(item, amount):
    conn = sqlite3.connect("my_ledger.db")
    cursor = conn.cursor()
    cursor.execute("INSERT INTO expenses (item, amount) VALUES (?, ?)", (item, amount))
    conn.commit()
    conn.close()
    return f"已记账：{item}，花费 {amount} 元。"

def get_total_expense():
    conn = sqlite3.connect("my_ledger.db")
    cursor = conn.cursor()
    cursor.execute("SELECT SUM(amount) FROM expenses")
    total = cursor.fetchone()[0]
    conn.close()
    return f"当前总共花费：{total} 元。" if total else "还没有任何记账记录。"
# 2. 真实的 Python 工具执行逻辑
def chat_with_ai(user_message, history):
    if not user_message:
        return "", history

    messages = [
        {"role": "system", "content": f"你是一个智能助手。{tools_description}"},
        {"role": "user", "content": user_message}
    ]
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json; charset=utf-8"
    }

    try:
        # ==========================================
        # 🌟 第 1 步：发送第一轮请求，获取 ai_reply
        # ==========================================
        data = {"model": "deepseek-chat", "messages": messages}
        json_data = json.dumps(data, ensure_ascii=False)
        response = requests.post(url, headers=headers, data=json_data.encode('utf-8'), timeout=30)
        result = response.json()
        
        # 这里就是生成 ai_reply 的地方！看看你代码里有没有这两行！
        ai_reply = result["choices"][0]["message"]["content"] 

        # ==========================================
        # 🌟 第 2 步：检查是否要调用工具
        # ==========================================
        if "tool" in ai_reply and "{" in ai_reply:
            try:
                # 提取 JSON 的逻辑
                start_idx = ai_reply.find("{")
                end_idx = ai_reply.rfind("}") + 1
                json_str = ai_reply[start_idx:end_idx]
                
                tool_json = json.loads(json_str)
                tool_name = tool_json.get("tool")
                args = tool_json.get("args", {})
                
                tool_result = execute_tool(tool_name, args)
                
                # 发送第二轮请求
                messages.append({"role": "assistant", "content": json_str})
                messages.append({"role": "user", "content": f"工具执行结果：{tool_result}\n请根据这个结果回答我最初的问题。"})
                
                data = {"model": "deepseek-chat", "messages": messages}
                response = requests.post(url, headers=headers, data=json.dumps(data, ensure_ascii=False).encode('utf-8'))
                result = response.json()
                final_reply = result["choices"][0]["message"]["content"]
                
                process_msg = f"（🔧 调用了工具：{tool_name}，结果：{tool_result}）\n\n"
                final_reply = process_msg + final_reply
            except Exception as e:
                final_reply = f"（❌ JSON 解析失败：{ai_reply}，报错：{e}）"
        else:
            final_reply = ai_reply

        history.append({"role": "user", "content": user_message})
        history.append({"role": "assistant", "content": final_reply})
        
        return "", history
        
    except Exception as e:
        return f"出错了: {e}", history

# 4. 搭建网页
with gr.Blocks(title="Agent 助手") as demo:
    gr.Markdown("# 🤖 拥有联网与时间能力的 Agent")
    gr.Markdown("你可以问我：'现在几点了？' 或者 '帮我搜索一下今天的AI新闻。'")
    
    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="输入你的问题...")
    msg.submit(chat_with_ai, [msg, chatbot], [msg, chatbot])

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Missing file: C:\Users\Administrator\.cache\huggingface\gradio\frpc\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: C:\Users\Administrator\.cache\huggingface\gradio\frpc
